In [ ]:
import math
import os
import random
import re
import shutil
from collections.abc import Callable, Sequence
from typing import Any, Literal

import cv2
import lightning as pl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
import segmentation_models_pytorch as smp
import torch
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as TF
from lightning import LightningDataModule, LightningModule
from PIL import Image
from skimage.metrics import structural_similarity
from sklearn.model_selection import GroupShuffleSplit
from torch import Tensor
from torch.utils.data import DataLoader, Dataset
from torchmetrics import Accuracy, ConfusionMatrix, F1Score, MaxMetric, MeanMetric
from torchvision import tv_tensors
from torchvision.io import read_image
from torchvision.ops import masks_to_boxes
from torchvision.transforms import InterpolationMode
from torchvision.utils import save_image
from tqdm.notebook import tqdm

root = rootutils.setup_root(search_from=".", indicator=".project-root", pythonpath=True)

from src.data.components.dataset import (
    FetalBrainPlanesDataset,
    USVideosFrameDataset,
    VideosFrameDataset,
)
from src.data.components.transforms import PadToAspectRatio, Resize
from src.data.utils.utils import (
    read_image_tensor,
    save_image_tensor,
    show_pytorch_images,
)
from src.models.head_segmentation_module import HeadSegmentationLitModule

data_dir = root / "data"
dataset_name = "FETAL_BRAIN_PLANES"
dataset_dir = data_dir / dataset_name

# Test model

In [ ]:
def predict_head_mask(model: HeadSegmentationLitModule, image: torch.Tensor) -> tuple[torch.Tensor, int]:
    """Binary mask [1, H, W] and frame label (same rules as HeadSegmentationLitModule.calculate_prediction)."""
    logits = model(image.unsqueeze(0))
    binary_mask, prediction_labels = model.calculate_hard_prediction(logits)
    return binary_mask.squeeze(0), int(prediction_labels.squeeze(0).item())


# checkpoint_file = root / "logs" / "head_segmentation_train" / "runs" / "2025-08-25_13-36-03" # cerulean-wave-694 42/8 - best
checkpoint_file = root / "logs" / "head_segmentation_train" / "runs" / "2025-10-13_15-27-50"
checkpoint = sorted(checkpoint_file.glob("checkpoints/epoch_*.ckpt"))[-1]
model = HeadSegmentationLitModule.load_from_checkpoint(str(checkpoint))
# disable randomness, dropout, etc...
model.eval()
model.to("cpu")
print("Loaded")

#### Test segmentation model

In [ ]:
head_segmentation = pd.read_csv(f"{data_dir}/FETAL_HEAD_SEGMENTATION/data.csv")
fetal_planes = pd.read_csv(f"{dataset_dir}/data.csv")

brain_images = []
brain_high_score_images = []
brain_low_score_images = []
unidentified_images = []
misidentified_images = []
# for index, plane_row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes), desc="Process images"):
#     for _, row  in head_segmentation[head_segmentation["Image_name"] == plane_row["Image_name"]].iterrows():
#         if row["Valid"] != 1:
#             continue

jnu_df = pd.read_csv(f"{data_dir}/FETAL_HEAD_SEGMENTATION/JNU-IFM/data.csv")
for index, row in tqdm(jnu_df.iterrows(), total=len(jnu_df), desc="Process images"):

    is_brain_plane = row["Brain_plane"] == 1
    # image_name = row["Image_name"]
    # image_path = images_dir / f"{image_name}.png"
    image_path = f"{data_dir}/FETAL_HEAD_SEGMENTATION/{row["Ultrasound_path"]}"
    image = read_image_tensor(image_path)

    with torch.no_grad():
        brain_plane = transform(image)
        prediction, prediction_label = predict_head_mask(model, brain_plane)

        if prediction_label == 1:
            brain_plane = pad_to_aspect_ration(image)
            angle = find_angle(prediction)
            prediction = TF.resize(prediction, size=brain_plane.shape[1:], interpolation=T.InterpolationMode.NEAREST)
            brain_plane = overlap_mask(brain_plane, prediction)

            prediction = TF.rotate(prediction, angle=angle, expand=True, interpolation=T.InterpolationMode.NEAREST)

            x1, y1, x2, y2 = masks_to_boxes(prediction)[0].int()
            perfect_mask = create_ellipse_tensor(
                prediction.shape[1],
                prediction.shape[2],
                x1 + ((x2 - x1) / 2),
                y1 + ((y2 - y1) / 2),
                (x2 - x1) / 2,
                (y2 - y1) / 2,
            )
            dice_score = get_dice_score(prediction, perfect_mask)

            if is_brain_plane:
                brain_images.append((index, brain_plane))

                if dice_score > 0.95:
                    brain_high_score_images.append((index, brain_plane))
                else:
                    brain_low_score_images.append((index, brain_plane))

            else:
                misidentified_images.append((index, brain_plane))
        else:
            if is_brain_plane:
                unidentified_images.append((index, image))


print(f"Number of brain images {len(brain_images)}")
print(f"Number of high score brain images {len(brain_high_score_images)}")
print(f"Number of low score brain images {len(brain_low_score_images)}")
print(f"Number of unidentified images {len(unidentified_images)}")
print(f"Number of misidentified images {len(misidentified_images)}")

Random brain images

In [ ]:
images = []
for rand_i in torch.randperm(len(brain_images))[:64]:
    i, image = brain_images[rand_i]
    images.append(image)
    # image_name = fetal_planes.Image_name[i]
    # plane = fetal_planes.Plane[i]
    # brain_plane = fetal_planes.Brain_plane[i]
    # subset = fetal_planes.Subset[i]
    # images.append((image, f"Brain_{brain_plane} - {subset}"))

if images:
    show_pytorch_images(images, gray=False).show()

Low score brain images

In [ ]:
images = []
for i, image in brain_low_score_images[:64]:
    # image_name = fetal_planes.Image_name[i]
    # image_path = images_dir / f"{image_name}.png"
    # image_original = read_image_tensor(image_path)
    # image_original = pad_to_aspect_ration(image_original)
    # plane = fetal_planes.Plane[i]
    # brain_plane = fetal_planes.Brain_plane[i]
    # subset = fetal_planes.Subset[i]
    # images.append((image_original, f"{i}:{brain_plane}"))
    images.append(image)

if images:
    show_pytorch_images(images, gray=False).show()

Unidentified brain images

In [ ]:
images = []
for i, image in unidentified_images[:64]:
    image = pad_to_aspect_ration(image)
    # image_name = fetal_planes.Image_name[i]
    # plane = fetal_planes.Plane[i]
    # brain_plane = fetal_planes.Brain_plane[i]
    # subset = fetal_planes.Subset[i]
    # images.append((image, f"{i}:Brain_{brain_plane} - {subset}"))
    # print(f"{i}: {image_name}")
    images.append(image)

if images:
    show_pytorch_images(images).show()

Not a brain images identify as brain image

In [ ]:
images = []
for i, image in misidentified_images:
    image_name = fetal_planes.Image_name[i]
    plane = fetal_planes.Plane[i]
    brain_plane = fetal_planes.Brain_plane[i]
    subset = fetal_planes.Subset[i]
    images.append((image, f"{i}:{plane}"))
    print(f'"{image_name}",')

if images:
    show_pytorch_images(images, gray=False).show()

Score for Not a brain images identify as brain image

In [ ]:
score_00_images = []
score_95_images = []
for i, _ in misidentified_images:
    plane = fetal_planes.Plane[i]
    image_name = fetal_planes.Image_name[i]
    image_path = images_dir / f"{image_name}.png"
    image = read_image_tensor(image_path)

    with torch.no_grad():
        brain_plane_tensor = transform(image)
        prediction, prediction_label = predict_head_mask(model, brain_plane_tensor)

        if prediction_label == 1:
            brain_plane = pad_to_aspect_ration(image)
            angle = find_angle(prediction)
            prediction = TF.resize(prediction, size=brain_plane.shape[1:], interpolation=T.InterpolationMode.NEAREST)
            brain_plane = overlap_mask(brain_plane, prediction)

            prediction = TF.rotate(prediction, angle=angle, interpolation=T.InterpolationMode.NEAREST)

            x1, y1, x2, y2 = masks_to_boxes(prediction)[0].int()
            perfect_mask = create_ellipse_tensor(
                prediction.shape[1],
                prediction.shape[2],
                x1 + ((x2 - x1) / 2),
                y1 + ((y2 - y1) / 2),
                (x2 - x1) / 2,
                (y2 - y1) / 2,
            )
            dice_score = get_dice_score(prediction, perfect_mask)

            if dice_score > 0.95:
                score_95_images.append((brain_plane, f"{dice_score.item():.3f}"))
            else:
                score_00_images.append((image, f"{i}:{plane}"))
                score_00_images.append((brain_plane, f"{dice_score.item():.3f}"))

print(f"Number of 0.00 {len(score_00_images) // 2}/{len(misidentified_images)}")
print(f"Number of 0.95 {len(score_95_images)}/{len(misidentified_images)}")

if score_00_images:
    show_pytorch_images(score_00_images, gray=False).show()
if score_95_images:
    show_pytorch_images(score_95_images, gray=False).show()

#### Crop images

In [ ]:
fetal_planes = pd.read_csv(dataset_dir / "data.csv")

if "Image_crop_name" not in fetal_planes.columns:
    fetal_planes.insert(1, "Image_crop_name", "")
else:
    fetal_planes["Image_crop_name"] = ""

brain_images = []
unidentified_images = []
misidentified_images = []

for index, row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes), desc="Process images"):
    plane = row["Plane"]
    image_name = row["Image_name"]
    image_path = images_dir / f"{image_name}.png"
    image = read_image_tensor(image_path)

    with torch.no_grad():
        brain_plane_tensor = transform(image)
        prediction, prediction_label = predict_head_mask(model, brain_plane_tensor)

        # Head detected (label matches HeadSegmentationLitModule training)
        if prediction_label == 1:
            brain_plane = pad_to_aspect_ration(image)

            angle = find_angle(prediction)
            prediction = TF.resize(prediction, size=brain_plane.shape[1:], interpolation=T.InterpolationMode.NEAREST)
            prediction = TF.rotate(prediction, angle=angle, expand=True, interpolation=T.InterpolationMode.NEAREST)
            brain_plane = TF.rotate(brain_plane, angle=angle, expand=True, interpolation=T.InterpolationMode.BILINEAR)

            x1, y1, x2, y2 = masks_to_boxes(prediction)[0].int()
            perfect_mask = create_ellipse_tensor(
                prediction.shape[1],
                prediction.shape[2],
                x1 + ((x2 - x1) / 2),
                y1 + ((y2 - y1) / 2),
                (x2 - x1) / 2,
                (y2 - y1) / 2,
            )
            dice_score = get_dice_score(prediction, perfect_mask)

            image = crop(brain_plane, x1, y1, x2, y2, pad=5)
            output_path = dataset_dir / "Images" / f"{image_name}_crop.png"
            save_image_tensor(image, output_path)
            fetal_planes.loc[index, "Image_crop_name"] = output_path.stem

            if plane == "Fetal brain":
                brain_images.append((index, dice_score.item()))
            else:
                misidentified_images.append((index, dice_score.item()))
        else:
            if plane == "Fetal brain":
                unidentified_images.append(index)

print(f"Number of unidentified images {len(unidentified_images)}")
print(f"Number of misidentified images {len(misidentified_images)}")
fetal_planes

#### Identify images training relevant

By default, all images are training relevant

In [ ]:
fetal_planes["Identified"] = 1

Verify low score brain images

In [ ]:
images = []
image_names = []
for i, dice_score in brain_images:
    if dice_score <= 0.95:
        plane = fetal_planes.Plane[i]
        brain_plane = fetal_planes.Brain_plane[i]
        subset = fetal_planes.Subset[i]

        image_name = fetal_planes.Image_name[i]
        image_path = images_dir / f"{image_name}.png"
        image = read_image_tensor(image_path)
        images.append((image, f"{i}: {brain_plane}"))
        image_names.append(image_name)

        image_name = fetal_planes.Image_crop_name[i]
        image_path = images_dir / f"{image_name}.png"
        image = read_image_tensor(image_path)
        images.append((image, f"{dice_score:.3f}"))
        image_names.append(image_name)

print(f"Number of low score brain images {len(images) // 2}")
if images:
    show_pytorch_images(images).show()

# for image_name in image_names:
#     print(f"\"{image_name}\"")

Verify unidentified brain images

In [ ]:
images = []
image_names = []
for i in unidentified_images:
    plane = fetal_planes.Plane[i]
    brain_plane = fetal_planes.Brain_plane[i]
    subset = fetal_planes.Subset[i]

    image_name = fetal_planes.Image_name[i]
    image_path = images_dir / f"{image_name}.png"
    image = read_image_tensor(image_path)
    images.append((image, f"{i}: {brain_plane}"))
    image_names.append(image_name)

    image_name = fetal_planes.Image_name[i]
    image_path = images_dir / f"{image_name}_crop.png"
    if image_path.exists():
        image = read_image_tensor(image_path)
        images.append((image, f"{i}: {brain_plane}"))
        image_names.append(image_name)
    else:
        images.append(None)

print(f"Number of unidentified brain images {len(images) // 2}")
if images:
    show_pytorch_images(images).show()

# for image_name in image_names:
#     print(f"\"{image_name}\"")

In [ ]:
for i in unidentified_images:
    image_name = fetal_planes.Image_name[i]
    fetal_planes.loc[i, "Image_crop_name"] = f"{image_name}_crop"

Remove Other brain images that are not showing brain

In [ ]:
remove_images = [1886, 1887, 5749, 7908, 8176, 11352]
for i in remove_images:
    print(i, fetal_planes.Image_name[i])
    fetal_planes.loc[i, "Identified"] = 0
    fetal_planes.loc[i, "Image_crop_name"] = ""

Not a brain images identify as brain image

In [ ]:
images = []
image_names = []
for i, dice_score in misidentified_images:
    plane = fetal_planes.Plane[i]
    brain_plane = fetal_planes.Brain_plane[i]
    subset = fetal_planes.Subset[i]

    image_name = fetal_planes.Image_name[i]
    image_path = images_dir / f"{image_name}.png"
    image = read_image_tensor(image_path)
    images.append((image, f"{i}: {plane}"))
    image_names.append(image_name)

    image_name = fetal_planes.Image_crop_name[i]
    image_path = images_dir / f"{image_name}.png"
    image = read_image_tensor(image_path)
    images.append((image, f"{dice_score:.3f}"))
    image_names.append(image_name)

print(f"Number of misidentified images {len(images) // 2}")
if images:
    show_pytorch_images(images).show()

# for image_name in image_names:
#     print(f"\"{image_name}\"")

Classify Other images as Other brain images that are showing brain image

In [ ]:
if "Brain_plane_fix" not in fetal_planes.columns:
    fetal_planes.insert(5, "Brain_plane_fix", "")
fetal_planes["Brain_plane_fix"] = fetal_planes["Brain_plane"]

for i, _ in misidentified_images:
    image_name = fetal_planes.Image_name[i]
    plane = fetal_planes.Plane[i]
    if plane == "Other":
        fetal_planes.loc[i, "Brain_plane_fix"] = "Other"
        fetal_planes.loc[i, "Image_crop_name"] = f"{image_name}_crop"
    else:
        fetal_planes.loc[i, "Image_crop_name"] = ""

remove_images = [8197]
for i in remove_images:
    fetal_planes.loc[i, "Brain_plane_fix"] = ""
    fetal_planes.loc[i, "Image_crop_name"] = ""

fetal_planes

In [ ]:
fetal_planes["Brain_plane"].value_counts()

In [ ]:
fetal_planes[fetal_planes["Identified"] == 1]["Brain_plane_fix"].value_counts()

In [ ]:
def print_subset_brain_planes(df, subset):
    print(subset)
    df = df[df["Identified"] == 1]
    df = df[df["Subset"] == subset]
    print(df["Brain_plane_fix"].value_counts())
    print()


print_subset_brain_planes(fetal_planes, "train")
print_subset_brain_planes(fetal_planes, "val")
print_subset_brain_planes(fetal_planes, "test")

In [ ]:
fetal_planes.to_csv(dataset_dir / "data.csv", index=False)
fetal_planes

In [ ]:
def extend_mean_median_plot(ax, mean, median, title):
    ax.axvline(mean, color="r", linestyle="dashed", linewidth=2)
    ax.axvline(median, color="b", linestyle="dashed", linewidth=2)
    ax.legend([f"mean {mean}", f"median {median}"], loc="upper left")
    ax.set_title(title, fontsize=14)


def print_resolution(scale, width):
    height = width / scale
    print(f"{height:.2f} / {width}")


data = []
for index, row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes), desc="Read images shape"):
    image_name = row["Image_name"]
    image_path = dataset_dir / "Images" / f"{image_name}.png"
    image = read_image(image_path)
    data.append((image.shape[0], image.shape[1], image.shape[2]))

df_shape = pd.DataFrame(data=data)
mean = df_shape.mean()
median = df_shape.median()

axs = df_shape.hist(column=[1, 2], bins=100, figsize=(20, 5))
extend_mean_median_plot(axs[0][0], mean[1], median[1], "Height")
extend_mean_median_plot(axs[0][1], mean[2], median[2], "Width")

scale = median[2] / median[1]
print_resolution(scale, 64)  # 64 / 64
print_resolution(scale, 128)  # 96 / 128
print_resolution(scale, 256)  # 192 / 256

In [ ]:
def select_ssim_frames(video: VideosFrameDataset, ssim_max: float):
    frames = []
    img = None
    for i, frame in enumerate(tqdm(video, desc="Video", leave=False)):
        frame_np = frame.numpy()[0]
        if img is None or structural_similarity(img, frame_np, data_range=frame_np.max() - frame_np.min()) <= ssim_max:
            frames.append((frame, i))
            img = frame_np
        break
    return frames


def select_ssim_videos(videos: USVideosFrameDataset, ssim_dataset_path, ssim_max):
    if not ssim_dataset_path.is_dir():
        ssim_dataset_path.mkdir(parents=True)

    ssim_images_path = ssim_dataset_path / "images"
    shutil.rmtree(ssim_images_path, ignore_errors=True)

    for video in tqdm(videos, desc="Videos"):
        ssim_frames_path = ssim_images_path / video.video_path.stem
        ssim_frames_path.mkdir(parents=True)
        frames = select_ssim_frames(video, ssim_max)
        for frame, frame_idx in frames:
            frame_path = ssim_frames_path / f"frame_{frame_idx:06d}.jpg"
            save_image(frame, str(frame_path))


# show first video
# frames = select_ssim_frames(videos[0], 0.6)
# show_pytorch_images(frames[:9])
# "ssim frames"

# select_ssim_videos(videos=videos, ssim_dataset_path=root / "data" / "US_VIDEOS_ssim_0.6", ssim_max=0.6)

In [ ]:
class StabilizeVideoEma:
    def __init__(self, alpha: float = 0.5):
        self.alpha = alpha
        self.args = None
        self.int_type = False

    def stabilize(self, *args):
        if self.args is None:
            self.args = self._extract_item_from_tensor(args)

            for arg in self.args:
                if isinstance(arg, int):
                    self.int_type = True
        else:
            for i in range(len(self.args)):
                self.args[i] = self.alpha * args[i] + (1 - self.alpha) * self.args[i]

        rs = self._extract_item_from_tensor(self.args)
        rs = [int(round(arg)) if self.int_type else arg for arg in rs]

        if len(rs) == 1:
            return rs[0]
        else:
            return rs

    def _extract_item_from_tensor(self, args):
        return [args[i].item() if isinstance(args[i], torch.Tensor) else args[i] for i in range(len(args))]

    def reset(self):
        self.args = None
        self.int_type = False


ssim_dataset_path = data_dir / "US_VIDEOS_ssim_0.7"
ssim_crop_dataset_path = data_dir / "US_VIDEOS_ssim_0.7_crop"

ssim_videos_path = ssim_dataset_path / "images"
ssim_crop_videos_path = ssim_crop_dataset_path / "images"

shutil.rmtree(ssim_crop_dataset_path)
ssim_crop_dataset_path.mkdir(parents=True)
ssim_crop_videos_path.mkdir()


alpha = 0.1
stabilizer_rotate = StabilizeVideoEma(alpha=alpha)
stabilizer_crop = StabilizeVideoEma(alpha=alpha)

saved_images = 0
for video_dir in tqdm(list(ssim_videos_path.iterdir()), desc="Videos"):

    output_dir = ssim_crop_videos_path / video_dir.name
    output_dir.mkdir(parents=True)

    for image_path in tqdm(list(video_dir.iterdir()), desc="Video images", leave=False):
        image = read_image(image_path)
        if image.shape[0] == 1:
            image = image.repeat(3, 1, 1)
        elif image.shape[0] == 4:
            image = image[:3, :, :]

        with torch.no_grad():
            brain_plane_tensor = transform(image)
            prediction, prediction_label = predict_head_mask(model, brain_plane_tensor)

            if prediction_label == 1:
                brain_plane = pad_to_aspect_ration(image)

                angle = find_angle(prediction)
                # angle = stabilizer_rotate.stabilize(angle)
                prediction = TF.resize(
                    prediction, size=brain_plane.shape[1:], interpolation=T.InterpolationMode.NEAREST
                )
                prediction = TF.rotate(prediction, angle=angle, expand=True, interpolation=T.InterpolationMode.NEAREST)
                brain_plane = TF.rotate(
                    brain_plane, angle=angle, expand=True, interpolation=T.InterpolationMode.BILINEAR
                )

                x1, y1, x2, y2 = masks_to_boxes(prediction)[0].int()
                # x1, y1, x2, y2 = stabilizer_crop.stabilize(x1, y1, x2, y2)
                image = crop(brain_plane, x1, y1, x2, y2, pad=5)

                output_path = output_dir / image_path.name
                image = image.permute(1, 2, 0).numpy()
                image = Image.fromarray(image)
                image.save(output_path)
                saved_images += 1
            else:
                stabilizer_rotate.reset()
                stabilizer_crop.reset()
    # break


print(f"Number of saved images {saved_images}")